# Real-Estate Data Collection Pipeline

An object-oriented Python pipeline for collecting residential real-estate data. It demonstrates:

- **HTTP requests** (`requests`) with realistic browser headers to fetch live listing pages
- **HTML parsing** (`BeautifulSoup`) to extract prices and structured `JSON-LD` `PostalAddress` records from Redfin listing cards
- **Geocoding** via the Google Geocoding API (address -> coordinates)
- **Geotagged photo retrieval** via the Flickr API around a coordinate
- **CLI argument parsing** (`argparse`) for reusable scripts

> API keys are placeholders. Supply your own Google Geocoding and Flickr keys to run the live cells.


1. Learn how to use requests library.
2. Use Google Place API for geocoding.
3. Use Beautiful Soup to collect house price.

# Requests.Get

In [ ]:
import requests

In [ ]:
# Send a request to a website
response = requests.get(url='https://www.google.com/')
print(response.text)

In [ ]:
# Save the responsive text as an html file
with open('output/google.html', 'w') as html_file:
    html_file.write(response.text)

In [ ]:
# Send a request with parameters
params = {
    "q": "madison"
}
response = requests.get(url='https://www.google.com/search', params=params)
print(response.url)

with open('output/google.html', 'w') as html_file:
    html_file.write(response.text)

In [ ]:
# Send a request to an image
response = requests.get(url='https://live.staticflickr.com/65535/49307295822_c047043e0a_c.jpg')

# Save the responsive content as an image
with open('output/google.html', 'wb') as img:
    img.write(response.content)

In [ ]:
# Close the response
response.close()

# Requests.Post

In [ ]:
# Send a post to a website
# https://httpbin.org/forms/post
data = {
    "custname": "Tang",
    "custtel": "608",
    "custemail": "user@example.com"
}
with requests.post('https://httpbin.org/post', data = data) as response:
    with open('output/posts.json', 'wb') as post_json:
        post_json.write(response.content)

# Flickr

In [ ]:
# Send a post to Flickr
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/41.0.2228.0 Safari/537.36'}
data = {
    "method": "flickr.photos.search",
    "enable_api_key": "on",
    "param_api_key": "YOUR_FLICKR_API_KEY",
    "enable_lat": "on",
    "param_lat": "43.07586",
    "enable_lon": "on",
    "param_lon": "-89.40114",
    "enable_radius": "on",
    "param_radius": "1",
    "format": "json-nc",
    "sign_call": "none"
}

with requests.post('https://www.flickr.com/services/api/explore/flickr.photos.search', data=data, headers=headers) as response:
    print(response.url)
    with open('output/Flickr.html', 'wb') as flickr_json:
        flickr_json.write(response.content)

In [ ]:
# Collect data
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/41.0.2228.0 Safari/537.36'}
data = {
    "method": "flickr.photos.search",
    "api_key": "YOUR_FLICKR_API_KEY",
    "lat": "43.07586",
    "lon": "-89.40114",
    "radius": "1",
    "format": "json",
    "nojsoncallback": "1"
}

url = 'code'

with requests.get('https://www.flickr.com/services/rest/', params=data, headers=headers) as response:
    print(response.url)
    with open(url+'/code/Flickr_data.html', 'wb') as flickr_json:
        flickr_json.write(response.content)

# Geocoding

In [ ]:
import json
import getpass

In [ ]:
api_key = getpass.getpass() # Enter your API key here

In [ ]:
# api_key = '' # This is for invaliding your API

In [ ]:
# Set the parameters of the request
params = {
    "address": "910 Eagle Heights, Madison, WI",
    "key": api_key,
    #"lang": "en"
}

response = requests.get(url="https://maps.googleapis.com/maps/api/geocode/json", params=params)

In [ ]:
print(response.url) # check the request url

In [ ]:
# Get the coordinates from the response
data = response.text
print(json.loads(data))
print("Coordinates of the place:", json.loads(data)["results"][0]["geometry"]["location"])

# Web Scraping with BeautifulSoup

In [ ]:
from bs4 import BeautifulSoup
import re

In [ ]:
# Send a request to the REDFIN to get house information
headers = {
    # Add headers in case the system identify you as a robot
    'User-Agent': 'Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/41.0.2228.0 Safari/537.36'
}
response = requests.get(url="https://www.redfin.com/zipcode/53703", headers=headers)
with open(url+'/code/house.html', 'w') as house_file:
    house_file.write(response.text)
    html = response.text

In [ ]:
with open(url+'/code/house.html', 'r') as html:
    html = html.read()

In [ ]:
# Parse the html file
soup = BeautifulSoup(html, 'html.parser')

In [ ]:
# Get the information of houses from the html
houses = soup.find_all("div", class_=re.compile(r'HomeCardContainer'))

if not houses:
    print("No house containers found with 'HomeCardContainer' in class name.")

for house in houses:
    # Try to find price using a less specific class name or attribute
    price_element = house.find('span', class_=re.compile(r'Price')) # Search for 'Price' in the class name
    if price_element and price_element.string:
        print(f"Price: {price_element.string.strip()}")
    else:
        print("Price not found for this house.")

In [ ]:
import json

# Helper function to find all PostalAddress dicts within a JSON object
def find_postal_addresses(obj):
    addresses = []
    if isinstance(obj, dict):
        if obj.get('@type') == 'PostalAddress':
            addresses.append(obj)
        for key, value in obj.items():
            addresses.extend(find_postal_addresses(value))
    elif isinstance(obj, list):
        for item in obj:
            addresses.extend(find_postal_addresses(item))
    return addresses

houses_data = []
script_tags = soup.find_all('script', type='application/ld+json')

for script in script_tags:
    try:
        json_data = json.loads(script.string)
        all_postal_addresses = find_postal_addresses(json_data)

        for address_info in all_postal_addresses:
            street_address = address_info.get('streetAddress')
            locality = address_info.get('addressLocality')
            region = address_info.get('addressRegion')
            postal_code = address_info.get('postalCode')
            country = address_info.get('addressCountry')

            if all([street_address, locality, region, postal_code, country]): # Ensure all parts are present
                full_address = f"{street_address}, {locality}, {region} {postal_code}, {country}"
                if {'Address': full_address} not in houses_data: # Avoid duplicates if found in multiple paths
                    houses_data.append({'Address': full_address})

    except json.JSONDecodeError:
        continue

if houses_data:
    for house in houses_data:
        print(f"Address: {house['Address']}")
else:
    print("No house addresses found using JSON-LD data.")

# Object-Oriented Programming

# Parse Arguments

In [ ]:
input_params_code = """
import argparse

parser = argparse.ArgumentParser(description='Process some parameters.')
parser.add_argument('-n', '--name', help='any variable')
parser.add_argument('-y', '--year', type=int, required=True, help='an integer number')
parser.add_argument('-g', '--gender', type=str, required=True, default='M', help='input M or F')

args = parser.parse_args()
print(args.name, args.year, args.gender)
"""

# Define the path where the file should be saved
file_path = url+'/InputParams.py'

# Write the code to the file
with open(file_path, 'w') as f:
    f.write(input_params_code)

print(f"File saved to: {file_path}")

In [ ]:
!python output/InputParams.py -n "Chen" -y 22 -g "F"

In [ ]:
!python output/InputParams.py --help

## Scraper Specification

Build a reusable web scraper (OOP) that, given an address, collects all listings in the same ZIP code. For each listing it saves price, address, coordinates (to CSV) and at least one photo. Run as: `python scraper.py -a "<address>"`.

## Scraper Specification

Build a reusable web scraper (OOP) that, given an address, collects all listings in the same ZIP code. For each listing it saves price, address, coordinates (to CSV) and at least one photo. Run as: `python scraper.py -a "<address>"`.